## Google Colab setup

Set `REPO_URL`, `BRANCH`, `RUN_NAME` and `BASE` in the next cell. Open this notebook from the same GitHub branch. The setup mounts Drive, clones code into the runtime and installs dependencies; local execution skips it. Use a fresh runtime after pushing new code. All notebooks in one experiment use the same run name; choose a new name for a fresh experiment. Run preprocessing and feature engineering to completion before model notebooks. The repository README describes the dataset layout and execution order.


In [1]:
# Configure this cell before running.
#   * Google Colab : edit REPO_URL / BRANCH / RUN_NAME / BASE in the Colab block.
#   * Local machine: edit LOCAL_RUN_NAME / LOCAL_BASE in the local block.
# Use the same RUN_NAME in every notebook of one experiment.
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    REPO_URL = "https://github.com/nguyenvanminh281005/bt4222-group12.git"
    BRANCH = "refactor-final"
    RUN_NAME = "run_01"
    BASE = Path("/content/drive/MyDrive/BT4222 - Group 12")
    CHECKOUT = Path("/content/bt4222_checkout")
    drive.mount("/content/drive")
    if not CHECKOUT.exists():
        subprocess.run([
            "git", "clone", "--branch", BRANCH, "--single-branch",
            REPO_URL, str(CHECKOUT)
        ], check=True)
    active_branch = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "branch", "--show-current"], text=True
    ).strip()
    if active_branch != BRANCH:
        raise ValueError("Checkout branch differs from BRANCH. Use a fresh Colab runtime.")
    CODE_DIR = (CHECKOUT if (CHECKOUT / "utils/data_utils.py").is_file()
                else CHECKOUT / "bt4222-group12")
    if not (CODE_DIR / "utils/data_utils.py").is_file():
        raise FileNotFoundError("The selected branch does not contain the Group 12 project.")
    os.environ.update({
        "BT4222_BASE": str(BASE),
        "MIND_DATA_DIR": str(BASE / "data"),
        "BT4222_PROCESSED_DIR": str(BASE / "processed"),
        "BT4222_FEATURE_DIR": str(BASE / "features"),
        "BT4222_GLOVE_PATH": str(BASE / "data/glove/glove.6B.300d.txt"),
        "BT4222_OUTPUT_DIR": str(BASE / "runs" / RUN_NAME),
    })
    os.chdir(CODE_DIR)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-r", "requirements.txt"
    ], check=True)
    print("Code:", CODE_DIR)
    print("Features:", os.environ["BT4222_FEATURE_DIR"])
    print("Run outputs:", os.environ["BT4222_OUTPUT_DIR"])

else:
    # ---- Local machine (VS Code / Jupyter on your own computer) ----
    # LOCAL_BASE must contain train/ and dev/ (news.tsv, behaviors.tsv,
    # entity_embedding.vec) and glove/glove.6B.300d.txt.
    LOCAL_RUN_NAME = "run_01"
    LOCAL_BASE = None  # None -> <repo>/data ; or e.g. Path("/path/to/data")

    CODE_DIR = next(
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "utils" / "data_utils.py").is_file()
    )
    BASE = Path(LOCAL_BASE) if LOCAL_BASE else CODE_DIR / "data"
    # setdefault: variables already exported in the shell take precedence.
    for key, value in {
        "BT4222_BASE": BASE,
        "MIND_DATA_DIR": BASE,
        "BT4222_PROCESSED_DIR": BASE / "processed",
        "BT4222_FEATURE_DIR": BASE / "processed" / "features",
        "BT4222_GLOVE_PATH": BASE / "glove" / "glove.6B.300d.txt",
        "BT4222_OUTPUT_DIR": CODE_DIR / "outputs" / LOCAL_RUN_NAME,
    }.items():
        os.environ.setdefault(key, str(value))
    os.environ.setdefault("BT4222_USE_DRIVE", "0")
    try:
        import torch
        device_name = (torch.cuda.get_device_name(0) if torch.cuda.is_available()
                       else "CPU only")
    except ImportError:
        device_name = "torch not installed"
    print("Mode: local |", device_name)
    print("Code:", CODE_DIR)
    print("Raw data:", os.environ["MIND_DATA_DIR"])
    print("Features:", os.environ["BT4222_FEATURE_DIR"])
    print("Run outputs:", os.environ["BT4222_OUTPUT_DIR"])

Mode: local | NVIDIA GeForce RTX 5060 Ti
Code: /mnt/data/KHTN2023/Project/bt4222-group12
Raw data: /mnt/data/KHTN2023/Project/bt4222-group12/data
Features: /mnt/data/KHTN2023/Project/bt4222-group12/data/processed/features
Run outputs: /mnt/data/KHTN2023/Project/bt4222-group12/outputs/run_01


# Feature Engineering

Construct aligned article, user, interaction and temporal representations from the processed partitions. Recommendation notebooks consume these artifacts directly. All array row conventions, feature lists and fitted statistics are exported with metadata.

## Feature definitions

**Content.** Encode title plus abstract with `sentence-transformers/all-MiniLM-L6-v2`. A missing abstract is replaced by its title. Article indices begin at 1, with a zero padding row. Content-based and MLP workflows read this same representation.

**Knowledge graph.** Average the available linked title/abstract TransE vectors, then normalize the result:

$$\bar{\mathbf k}_a=\frac{1}{|Q_a|}\sum_{q\in Q_a}\mathbf e_q,\qquad \mathbf k_a=\frac{\bar{\mathbf k}_a}{\|\bar{\mathbf k}_a\|_2}.$$

An article without mapped entities has a zero vector and `has_kg_entity=0`. Category/subcategory vocabularies are fitted on training-visible articles and reserve PAD/UNK. Learned categorical embedding layers belong to the model, not to the saved category IDs.

**History.** Build one profile per distinct supplied history. The default content profile is the normalized arithmetic mean; optional position weighting increases the weight of later clicks. Category proportions and Shannon entropy describe reading interests. An empty history has zero raw profile vectors/scalars; later standardization may make its scalar input nonzero.

**Historical popularity.** Count clicks and exposures from days strictly before the impression day. With training-period prior $\mu$, use

$$\mathrm{hist\_ctr}(a,d)=\frac{C_{a,<d}+\alpha\mu}{E_{a,<d}+\alpha},\qquad \alpha=20.$$

The setting for test-day statistics explicitly controls whether preceding validation-day observations are included. The prior is fitted on the training period. This is a daily historical score, not same-day online updating.

**Interactions and time.** Compute profile/candidate cosine similarities, category affinity, binary category match and KG similarity. Empty-history category match is zero; undefined similarity scalars use training means. Encode hour by sine/cosine and retain the existing optional return-habit statistic. Inactivity/context exports remain keyed by impression.

## Sequence inputs and exports

Title tokens, GloVe alignment and padded history indices supply trainable attention models. Previous-day category-stratified popularity sets are separate sequence inputs; their first-day fallback policy is explicit in the preparation code. They do not replace the common article content vectors.

Fit normalization on training rows only: $z=(x-\mu_{train})/\sigma_{train}$. Preserve binary flags. Export tables, arrays, fitted means/scales and field ordering; model-specific training simulation and loss remain in model notebooks. Rebuild representations when the input source or feature configuration changes.

The complete feature dictionary below defines the inputs, transformations, missing-value conventions and exported artifacts used in this notebook.


## Feature Dictionary and Preparation Contract

This dictionary documents the implemented preparation in the feature engineering workflow, its intermediate representations and exported inputs. The EDA notebook reports descriptive distributions separately. A feature's presence in the catalog does not mean every model consumes it.


### 1. Pipeline and identities

Preprocessing merges the train/development behavior sources, deduplicates news by `news_id`, parses timestamps, fills empty history strings, assigns global impression IDs and creates chronological 5–1–1 partition labels. Its full behavior table retains all source rows; model training groups can later be sampled or filtered.

| Field | Meaning and convention |
|---|---|
| `impression_id` | Globally unique processed impression identity. |
| `orig_impression_id`, `source` | Original source identity, retained for joins and traceability. |
| `news_id`, `news_idx` | Raw article identifier and one-based catalog array index. Row 0 is padding. |
| `news_id2idx` | In-memory article-to-index mapping aligned with the catalog and arrays. |
| `cbf_row_idx` | Zero-based catalog view: `news_idx - 1`; consumers omit the shared array's padding row. |
| `profile_idx` | Index of a distinct supplied history. Profile 0 is the empty history. |
| `day`, `split` | Calendar-day offset and train/validation/test membership. |
| `group_id`, `slot` | Ranking-group identity and candidate position in its prepared group. |
| `in_train` | Article appears in a training candidate slate or training history; used to fit category vocabularies. |

Partition days 0–4 are training, day 5 validation and day 6 test. Indices and flags are bookkeeping or preparation state, not automatically predictive features. Training vocabularies reserve PAD=0 and UNK=1; unseen held-out categories map to UNK.


### 2. Article representations

| Feature | Source | Definition | Shape / missing handling |
|---|---|---|---|
| `article_content_emb` | Title and abstract | SentenceTransformer `sentence-transformers/all-MiniLM-L6-v2`; concatenate title and abstract, replacing an absent/blank abstract with title; L2-normalized encoding. | Normally 384 dimensions per article; padding row is zero. |
| `kg_entity_emb` | Title/abstract entity annotations and entity vectors | Resolve distinct linked Wikidata IDs, average available TransE vectors and L2-normalize the nonzero mean. | Normally 100 dimensions; zero when no entity vector maps; padding is zero. |
| `has_kg_entity` | Entity-vector coverage | 1 when at least one linked entity has a usable vector, otherwise 0. | Binary; retained alongside the KG vector. |
| `category_idx` | Raw category | Training-visible category vocabulary index. | Integer including PAD/UNK; distinct from descriptive full-catalog IDs. |
| `subcategory_idx` | Raw subcategory | Training-visible subcategory vocabulary index. | Integer including PAD/UNK. |

For the $n_a$ mapped entities in $Q_a$:

$$\bar{\mathbf k}_a=\frac{1}{n_a}\sum_{q\in Q_a}\mathbf e_q,\qquad
\mathbf k_a=\frac{\bar{\mathbf k}_a}{\|\bar{\mathbf k}_a\|_2}.$$

A zero norm remains zero. Linked entity presence and successful vector mapping are different coverage measures.

Category/subcategory **embedding layers** are learned inside MLP models and both have width 32. Feature engineering exports their indices, not trained category vectors. NRMS title encoders learn a different representation from token sequences. Supplied `data/utils/embedding*.npy` arrays have not been verified as MiniLM article vectors and are not used as substitutes.


### 3. History and user-profile features

Let $H_u$ be the article indices resolved from the history supplied with an impression and $n_u$ its length. Profiles are keyed by distinct histories, allowing the same user to have different observed states over time.

| Feature | Definition | Empty-history behavior / use |
|---|---|---|
| `history_length` | Number of resolved article indices in the profile history. | 0; model reporting groups use 0 / 1–4 / at least 5. |
| `log_history_length` | $\log(1+n_u)$. | 0 before standardization. |
| `has_history` | $\mathbb{1}[n_u>0]$. | Binary 0. |
| `user_history_emb` | L2-normalized mean of history content vectors. `USER_POOLING='position'` optionally uses weights 1 through history length. | Zero raw vector for profile 0. |
| `user_category_affinity` | Category counts divided by the number of resolved history articles, aligned with the category vocabulary including special indices. | Zero vector; dimension is `N_CAT`, not a hard-coded 18. |
| `reading_entropy` | Shannon entropy of nonzero category proportions, using natural logarithms. | 0 for exported empty-history profiles. |
| `diversity_ratio` | Number of categories with nonzero history count divided by history length. | 0; optional scalar. |
| `user_last` | Normalized mean content vector of the latest `LAST_N=5` history articles. | Intermediate used by `cos_last5`; not a separately exported model input. |
| `user_kg` | Normalized mean of history article KG representations with mapped entities. | Intermediate; unavailable entity evidence produces undefined KG interaction similarity. |
| `top_cat` | Category index with maximum history count; ties follow `argmax` order. | Undefined category index 0; empty-history category match is explicitly 0. |

For proportions $p_c$:

$$H(u)=-\sum_{c:p_c>0}p_c\log p_c.$$

The prepared history content vector uses static pooling. Multi-head/candidate-conditioned attention is learned within the corresponding model; it is not implied by the name `user_history_emb`.


### 4. Historical popularity

Count exposures and clicked exposures before the impression day, using the full unsampled logs. The training CTR prior is

$$\mu_{train}=\frac{\text{training clicked exposures}}{\text{training exposures}}.$$

| Feature | Definition | Unseen-article behavior |
|---|---|---|
| `hist_ctr` | $(C_{a,<d}+\alpha\mu_{train})/(E_{a,<d}+\alpha)$, with `CTR_ALPHA=20`. | Receives the training prior when exposure count is zero. |
| `log_exposure` | $\log(1+E_{a,<d})$. | 0. |
| `is_cold_article` | $\mathbb{1}[C_{a,<d}=0]$, under the configured daily historical window. | 1, including articles exposed but never clicked. |

`STATS_USE_VAL_FOR_TEST=True` allows test-day features to use preceding validation-day observations; False freezes them at the end of train. Neither setting uses the current impression day's labels. The training prior is a fitted training statistic, not an event-by-event prior.

The standalone smoothed-popularity baseline applies this scoring policy to all users. The MLP fallback applies its historical-CTR ranking only to empty-history users. The advanced hybrid CTR scorer instead has its own declared build windows and validation-tuned alpha; its score is a model output, not an alias for `hist_ctr`.


### 5. Candidate–history interactions

| Feature | Definition | Missing treatment |
|---|---|---|
| `cos_user_hist` | Dot product of normalized full-history content profile and candidate content vector. | Undefined cold-profile interactions receive the training candidate mean. |
| `cos_last5` | Dot product of the normalized latest-five profile and candidate content vector. | Training candidate mean when undefined. |
| `max_cos_hist` | Maximum content cosine similarity between the candidate and any resolved history article. | Training candidate mean for empty history. |
| `cat_affinity` | User category-proportion entry at the candidate category index. | Undefined empty-history value is filled from training candidate statistics. |
| `category_match` | 1 when candidate category equals the most frequent history category, otherwise 0. | Explicitly 0 for empty history; remains binary. |
| `kg_match` | Dot product of normalized user KG profile and candidate KG representation. | Undefined if either side lacks mapped KG evidence; fill with training candidate mean. |

Scalar interaction features are calculated before final raw-vector zero filling. Consequently, a zero exported empty-history vector does not imply every previously undefined interaction scalar is zero. The presence flags identify missing evidence.


### 6. Time and impression context

| Feature | Definition | Handling |
|---|---|---|
| `hour_sin`, `hour_cos` | $\sin(2\pi h/24)$ and $\cos(2\pi h/24)$; primary feature arrays use hour plus minute/60. | Already cyclic; no additional scaling in the saved input assembly. |
| `habit_score` | $\cos(2\pi\Delta t/24)$, where $\Delta t$ is hours since the previous impression of the same user. | Training mean for first observed impression; optional feature. |
| `inactivity_gap` | Hours since that user's preceding recorded impression. | Undefined at the first impression; retained in the context/EDA exports. |
| `first_imp` | Indicator that no preceding impression exists. | Binary missingness indicator in the stacking context artifact. |

The stacking context artifact currently computes hour encodings from the integer hour, whereas primary arrays include minutes. These are distinct implemented granularities and must be identified when comparing or further consolidating context. Timestamps are used as recorded; the code does not establish a local-time-zone conversion.


### 7. Title and history sequences

| Prepared input | Definition |
|---|---|
| `news_title_lookup` | Lowercased regex tokenization; vocabulary count threshold 2 over the catalog; PAD=0, UNK=1; truncate/pad titles to 30 tokens. |
| `pretrained_embeddings` | 300-dimensional GloVe alignment with the token vocabulary; source random-normal initialization for unmatched words and zero padding row. |
| `user_history_indices.npy` | Per-profile last 50 article indices, right-padded with zeros for candidate-conditioned history attention. |
| NRMS record histories | Last 50 resolved articles, **left-padded** with zeros as required by the source NRMS workflow. |
| `pop_sets` | Previous-day category-stratified popularity sets: at least 20 daily category clicks, two slots per eligible category plus ten proportional extra slots. |
| `top_popular_ids` | Global top 20 article indices by five-day training impression clicks, for the separate hard-fallback reference. |

The first popularity-set day falls back to its own day's counts in the existing preparation. This boundary behavior is not strictly past-only and should not be described as such. Sequence padding differs by consumer and is part of the tensor contract; the underlying article mapping is common.


### 8. EDA-only descriptors

| Descriptor | Purpose |
|---|---|
| `category_id`, `subcategory_id` | Full-catalog descriptive encoding, distinct from training-fitted model indices. |
| `title_word_count`, `title_char_length` | Text-length summaries before encoding. |
| Abstract word/character counts and corpus vocabulary | Describe non-missing raw abstract text separately from imputation. |
| `title_kg_count`, `abs_kg_count`, `total_kg_count` | Count linked annotations; do not equate them with mapped-vector coverage. |
| Full-period empirical article CTR | Descriptive clicked/exposed ratio over merged logs, not a historical predictor. |
| `article_recency_observed` | Sampled exposure age relative to the article's first recorded exposure; not verified publication age. |
| `session_hour`, `session_dow`, `inactivity_bin` | Clock/date and return-gap distributions; bins are under 1h, 1–6h, 6–24h, at least 24h, plus new-user state. |
| `slate_length` | Number of exposed candidates in an impression. |

The model contract excludes `slate_length`, `article_recency` and `session_dow` from the selected MLP feature space. They remain useful for analysis. User-level statistics use the earliest recorded impression per user; event-level statistics retain every supplied history snapshot.


### 9. Standardization, sampling and artifacts

Continuous candidate/profile scalars use training-fitted means and standard deviations. Binary `is_cold_article`, `category_match` and `has_history` are not standardized. Hour sine/cosine retain their original scale; the optional habit score uses training-fitted statistics. Raw zero profiles can therefore have nonzero standardized scalar values.

Training ranking groups retain one positive and up to four sampled negatives from the same impression, without replacement. Evaluation retains full eligible slates. Counts for historical CTR are computed before sampling. Split rows and ranking groups are different units and must not share an unlabeled denominator.

| Artifact | Contents |
|---|---|
| `articles.*` | Catalog identities, category indices, entity and training-visibility flags. |
| `user_profiles.*` | Profile indices and user scalar features. |
| `{train,val,test}/impressions.*` | Impression identities, time, profile association and contextual scalars. |
| `{train,val,test}/groups.*`, `{train,val,test}/candidates.*` | Ranking boundaries, labels, candidate order and interactions. |
| `article_content_emb.npy`, `kg_entity_emb.npy` | Common padded article representations. |
| `user_history_emb.npy`, `user_category_affinity.npy` | Profile-aligned content and category vectors. |
| `mlp_inputs.pkl` | Ordered arrays and fitted normalization dictionaries. |
| `feature_meta.json` | Feature lists, dimensions, settings, means, temporal/cohort conventions and source signatures. |
| `feature_manifest.json` | Completed bundle identity, preparation settings and content hashes for exported files. |
| `hybrid_news.pkl`, `hybrid_user_state.pkl` | Catalog view and impression-keyed stacking context. |
| `nrms_features.pkl` | Title/word arrays, temporal records, popularity sets and mapping/configuration information. |

`.*` denotes Parquet where available and pickle otherwise. Preprocessing/EDA and popularity baselines have real-data outputs. Full feature generation and real-data neural training are separate execution steps; this catalog does not certify an unexecuted embedding artifact or reproduce every historical report statistic.


### 10. Validation boundaries

Training-only vocabularies and scalar transforms must be fitted before held-out use. Source signatures identify raw file contents independently of location. A final manifest identifies the preparation code/settings and verifies exported contents. Model consumers validate the prepared bundle without raw files; checkpoint output roots remain associated with that bundle. Raw-history and resolved-history lengths can differ when an ID is absent from the catalog; preserve evaluator provenance when interpreting cold cohorts.


In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "utils" / "data_utils.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_utils import project_paths, load_preprocessed, load_table, save_table

if os.environ.get("BT4222_USE_DRIVE") == "1":
    from google.colab import drive

    drive.mount("/content/drive")

PATHS = project_paths(PROJECT_ROOT)
from utils.data_utils import (source_fingerprint, validate_features, content_identity,
                             publish_features, bind_feature_run, file_digest)


In [3]:
import json
import re
import math
import time
import random
from collections import Counter, defaultdict
from types import SimpleNamespace
import numpy as np
import pandas as pd
import torch
from utils.data_utils import source_directory

FEATURE_DIR = PATHS["features"]
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
(FEATURE_DIR / "feature_manifest.json").unlink(missing_ok=True)
previous_meta = FEATURE_DIR / "feature_meta.json"

if previous_meta.exists():
    previous_source = json.loads(previous_meta.read_text()).get("source_files")
    if previous_source is not None and previous_source != source_fingerprint(PATHS):
        raise ValueError(
            "Raw sources changed. Choose a clean BT4222_FEATURE_DIR before rebuilding features."
        )

SEED = 42
np.random.seed(SEED)


In [4]:
import os, sys, json, math, time, gc
from collections import Counter
import numpy as np
import pandas as pd

DATA_ROOT = str(PATHS["raw"])
OUT_DIR = str(FEATURE_DIR)
SEED = 42
N_NEG = 4  # negatives per positive in training
CTR_ALPHA = 20.0  # smoothing strength for historical CTR
LAST_N = 5  # "last-5 clicks" short-term interest window
USER_POOLING = "mean"  # 'mean' or 'position' (linearly up-weights recent clicks)
SPARSE_USER_MAX = 5  # sparse user slice: 1 <= history_length < 5
STATS_USE_VAL_FOR_TEST = True  # test-day popularity uses days 1-6 (all strictly before test). False -> days 1-5 only
TEXT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMB_BATCH = 256
TRAIN_DAYS, VAL_DAY, TEST_DAY = [0, 1, 2, 3, 4], 5, 6
SPLITS = ["train", "val", "test"]
os.makedirs(OUT_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print("OUT_DIR =", OUT_DIR)


def save_table(df, name):
    """Parquet when pyarrow is available, pickle otherwise. load_table() reads either."""
    path = os.path.join(OUT_DIR, name)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path + ".parquet", index=False)
        return path + ".parquet"
    except ImportError:
        df.to_pickle(path + ".pkl")
        return path + ".pkl"


OUT_DIR = /mnt/data/KHTN2023/Project/bt4222-group12/data/processed/features


In [5]:
beh, news = load_preprocessed(PATHS)
entity_files = [
    str(source_directory(PATHS["raw"], source) / "entity_embedding.vec")
    for source in ["train", "dev"]
]
first_day = beh["time"].dt.normalize().min()
split_summary = beh.groupby("split").agg(
    impressions=("impression_id", "size"),
    users=("user_id", "nunique"),
    start=("time", "min"),
    end=("time", "max"),
)


## 1. Parse impressions, global article index & training-period vocabularies
* `news_idx` 0 is reserved for padding; real articles are 1..N.
* Category / subcategory vocabularies are built **only from articles seen in the training period** (training candidates + training histories). Index 0 = PAD, 1 = UNK. Categories first seen in val/test map to UNK, which the model trains through id-dropout.
* Raw MIND `category` (18) and `subcategory` (~270) are used, and the **same** category vocabulary is used on the user side and article side (shared table).


In [6]:
news["abstract"] = news["abstract"].fillna("")
news["title"] = news["title"].fillna("")
news_id2idx = {nid: i + 1 for i, nid in enumerate(news["news_id"])}
N_ART = len(news) + 1  # + PAD row
news["news_idx"] = np.arange(1, N_ART)


def parse_history(h):
    if not isinstance(h, str) or not h.strip():
        return np.zeros(0, dtype=np.int32)
    return np.array(
        [news_id2idx[x] for x in h.split() if x in news_id2idx], dtype=np.int32
    )


def parse_impressions(s):
    ids, labels = [], []
    for item in s.split():
        nid, lab = item.rsplit("-", 1)
        ids.append(news_id2idx[nid])
        labels.append(int(lab))
    return np.array(ids, dtype=np.int32), np.array(labels, dtype=np.int8)


beh["history"] = beh["history"].fillna("")
parsed = beh["impressions"].map(parse_impressions)
beh["cand_idx"] = parsed.map(lambda t: t[0])
beh["cand_lab"] = parsed.map(lambda t: t[1])
beh["n_pos"] = beh["cand_lab"].map(lambda a: int(a.sum()))
beh["n_cand"] = beh["cand_idx"].map(len)
beh = beh.drop(columns=["impressions"])
tr = beh[beh.split == "train"]
train_hist_idx = np.unique(
    np.concatenate(
        [parse_history(h) for h in tr["history"].unique()] + [np.zeros(0, np.int32)]
    )
)
train_cand_idx = np.unique(np.concatenate(tr["cand_idx"].tolist()))
in_train = np.zeros(N_ART, dtype=bool)
in_train[train_hist_idx] = True
in_train[train_cand_idx] = True
news["in_train"] = in_train[news["news_idx"].values]


def build_vocab(values):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for v in sorted(set(values)):
        vocab[v] = len(vocab)
    return vocab


cat_vocab = build_vocab(news.loc[news.in_train, "category"])
subcat_vocab = build_vocab(news.loc[news.in_train, "subcategory"])
news["category_idx"] = news["category"].map(cat_vocab).fillna(1).astype(int)
news["subcategory_idx"] = news["subcategory"].map(subcat_vocab).fillna(1).astype(int)
art_cat = np.zeros(N_ART, dtype=np.int32)
art_cat[news.news_idx.values] = news.category_idx.values
art_subcat = np.zeros(N_ART, dtype=np.int32)
art_subcat[news.news_idx.values] = news.subcategory_idx.values
N_CAT, N_SUBCAT = len(cat_vocab), len(subcat_vocab)
print(f"Articles: {N_ART-1:,} ({news.in_train.sum():,} seen in training period)")
print(
    f"Category vocab: {N_CAT} (incl. PAD/UNK) | subcategory vocab: {N_SUBCAT} (incl. PAD/UNK)"
)
print(
    "Articles mapped to UNK category:",
    int((news.category_idx == 1).sum()),
    "| UNK subcategory:",
    int((news.subcategory_idx == 1).sum()),
)
print("Impressions with 0 clicks:", int((beh.n_pos == 0).sum()))


Articles: 65,238 (47,538 seen in training period)
Category vocab: 19 (incl. PAD/UNK) | subcategory vocab: 263 (incl. PAD/UNK)
Articles mapped to UNK category: 1 | UNK subcategory: 9
Impressions with 0 clicks: 0


## 2. Content Semantics
### 2.1 `article_content_emb` — MiniLM 384-d on *title + abstract*

- all-MiniLM-L6-v2 encoder on one embedding of the concatenated text (title + abstract). Missing abstracts are filled with the title.
- Vectors are L2-normalised so dot product = cosine.
- Metadata only, no labels → no leakage.


In [7]:
EMB_PATH = os.path.join(OUT_DIR, "article_content_emb.npy")
EMB_IDS_PATH = os.path.join(OUT_DIR, "article_content_emb_ids.json")
texts = (
    news["title"]
    + ". "
    + np.where(news["abstract"].str.strip() == "", news["title"], news["abstract"])
).tolist()
embedding_settings = {"texts": texts, "text_model": TEXT_MODEL, "normalize": True}
embedding_key = content_identity(embedding_settings)
cache_info = FEATURE_DIR / "article_content_emb_config.json"
cached = False

if os.path.exists(EMB_PATH) and os.path.exists(EMB_IDS_PATH):
    with open(EMB_IDS_PATH) as f:
        cached = (json.load(f) == news["news_id"].tolist()
                  and cache_info.is_file()
                  and json.loads(cache_info.read_text()).get("identity") == embedding_key)

if cached:
    content_emb = np.load(EMB_PATH)
    print("Loaded cached article_content_emb", content_emb.shape)
else:
    import torch
    from sentence_transformers import SentenceTransformer

    device = "cuda" if torch.cuda.is_available() else "cpu"
    st = SentenceTransformer(TEXT_MODEL, device=device)
    enc = st.encode(
        texts,
        batch_size=EMB_BATCH,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    content_emb = np.vstack(
        [np.zeros((1, enc.shape[1]), np.float32), enc]
    )  # row 0 = PAD
    np.save(EMB_PATH, content_emb)
    with open(EMB_IDS_PATH, "w") as f:
        json.dump(news["news_id"].tolist(), f)
    cache_info.write_text(json.dumps({"identity": embedding_key}), encoding="utf-8")
    del st
    print("Encoded article_content_emb", content_emb.shape)

assert content_emb.shape[0] == N_ART
D_TEXT = content_emb.shape[1]


Loaded cached article_content_emb (65239, 384)


## 3. `kg_entity_emb` (TransE 100-d) + `has_kg_entity`

- Mean of the TransE vectors of all Wikidata entities in the title and abstract, L2-normalised.
- Articles without a linked entity get `has_kg_entity = 0`; their vector is zero.


In [8]:
entity_vecs = {}

for path in entity_files:
    if not os.path.exists(path):
        continue
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.split()
            if len(parts) > 2 and parts[0] not in entity_vecs:
                entity_vecs[parts[0]] = np.asarray(parts[1:], dtype=np.float32)

D_KG = len(next(iter(entity_vecs.values())))
print(f"TransE vectors loaded: {len(entity_vecs):,} ({D_KG}-d)")


def entity_ids(s):
    if not isinstance(s, str) or not s.strip():
        return []
    try:
        return [
            e["WikidataId"]
            for e in json.loads(s)
            if isinstance(e, dict) and "WikidataId" in e
        ]
    except (json.JSONDecodeError, TypeError):
        return []


kg_raw = np.full((N_ART, D_KG), np.nan, dtype=np.float32)

for idx, t_ent, a_ent in zip(
    news.news_idx, news.title_entities, news.abstract_entities
):
    vecs = [
        entity_vecs[e]
        for e in set(entity_ids(t_ent) + entity_ids(a_ent))
        if e in entity_vecs
    ]
    if vecs:
        v = np.mean(vecs, axis=0)
        n = np.linalg.norm(v)
        kg_raw[idx] = v / n if n > 0 else v

has_kg = ~np.isnan(kg_raw[:, 0])
has_kg[0] = False
news["has_kg_entity"] = has_kg[news.news_idx.values].astype(int)
print(f"Articles with >=1 linked KG entity: {has_kg[1:].mean():.2%}")


TransE vectors loaded: 31,451 (100-d)


Articles with >=1 linked KG entity: 87.03%


## 4. Popularity group (historical daily statistics)
For an impression on day *d*, using only impressions on days **< d** (full slates, not the sampled ones):
* `hist_ctr` = (clicks + α·prior) / (exposures + α), with α = `CTR_ALPHA` and prior = global CTR of the training period. Unseen articles get the prior, i.e. a low-confidence signal.
* `log_exposure` = log1p(exposures)
* `is_cold_article` = 1 if the article has **no clicks** before day *d* (contract cold-article definition).

With `STATS_USE_VAL_FOR_TEST = True`, test (day 7) uses days 1–6, which is still strictly before the test day. Set it to `False` to freeze the stats at the end of training.


In [9]:
day_arr = np.repeat(beh["day"].values, beh["n_cand"].values)
cand_flat = np.concatenate(beh["cand_idx"].tolist())
lab_flat = np.concatenate(beh["cand_lab"].tolist()).astype(np.int64)
exp_by_day = np.zeros((7, N_ART), dtype=np.int64)
clk_by_day = np.zeros((7, N_ART), dtype=np.int64)
np.add.at(exp_by_day, (day_arr, cand_flat), 1)
np.add.at(clk_by_day, (day_arr, cand_flat), lab_flat)
cum_exp = np.vstack([np.zeros((1, N_ART), np.int64), np.cumsum(exp_by_day, 0)[:-1]])
cum_clk = np.vstack([np.zeros((1, N_ART), np.int64), np.cumsum(clk_by_day, 0)[:-1]])
CTR_PRIOR = clk_by_day[TRAIN_DAYS].sum() / exp_by_day[TRAIN_DAYS].sum()
LAST_STATS_DAY = VAL_DAY if STATS_USE_VAL_FOR_TEST else TRAIN_DAYS[-1]


def stats_day(d):
    """Index into cum_* for an impression on day d (never includes day d itself)."""
    return np.minimum(d, LAST_STATS_DAY + 1)


del day_arr, cand_flat, lab_flat
print(f"Training-period CTR prior = {CTR_PRIOR:.4f}")

for d in range(7):
    sd = stats_day(d)
    print(
        f"day {d}: stats from days < {sd} | articles with >=1 prior click: {(cum_clk[sd] > 0).sum():,}"
    )


Training-period CTR prior = 0.0410
day 0: stats from days < 0 | articles with >=1 prior click: 0
day 1: stats from days < 1 | articles with >=1 prior click: 1,378
day 2: stats from days < 2 | articles with >=1 prior click: 2,283
day 3: stats from days < 3 | articles with >=1 prior click: 3,736
day 4: stats from days < 4 | articles with >=1 prior click: 5,046
day 5: stats from days < 5 | articles with >=1 prior click: 6,398
day 6: stats from days < 6 | articles with >=1 prior click: 7,713


## 5. User profiles
MIND's `history` field is the user's click history **before** the impression, so it is leak-free by construction. A profile is computed once per distinct history:
* `user_history_emb`: pooled `article_content_emb` of clicked articles (mean, or position-weighted), L2-normalised
* `history_length`, `log_history_length`, `has_history`
* `reading_entropy`: Shannon entropy of the history's category distribution
* `user_category_affinity`: share of history clicks per category, aligned with the **shared category vocabulary** (the model multiplies it by the article category embedding table)
* `diversity_ratio` (optional): unique categories / history length

Profile 0 is the empty history (cold user). Its exported content, category-affinity and scalar profile values are zero.


In [10]:
beh["profile_key"] = beh["history"].str.strip()
uniq_hist = pd.Index(beh["profile_key"].unique())

if "" in uniq_hist:
    uniq_hist = pd.Index([""] + [h for h in uniq_hist if h != ""])
else:
    uniq_hist = pd.Index([""] + list(uniq_hist))

beh["profile_idx"] = uniq_hist.get_indexer(beh["profile_key"])
P = len(uniq_hist)
hist_lists = [parse_history(h) for h in uniq_hist]
hist_len = np.array([len(h) for h in hist_lists], dtype=np.int32)
user_emb = np.full((P, D_TEXT), np.nan, dtype=np.float32)
user_last = np.full((P, D_TEXT), np.nan, dtype=np.float32)
user_kg = np.full((P, D_KG), np.nan, dtype=np.float32)
user_aff = np.full((P, N_CAT), np.nan, dtype=np.float32)
entropy = np.full(P, np.nan, dtype=np.float32)
diversity = np.full(P, np.nan, dtype=np.float32)
top_cat = np.zeros(P, dtype=np.int32)  # 0 = undefined (cold)


def l2n(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v


t0 = time.time()

for p, h in enumerate(hist_lists):
    if len(h) == 0:
        continue
    E = content_emb[h]
    if USER_POOLING == "position":
        w = np.arange(1, len(h) + 1, dtype=np.float32)
        user_emb[p] = l2n((w[:, None] * E).sum(0) / w.sum())
    else:
        user_emb[p] = l2n(E.mean(0))
    user_last[p] = l2n(
        E[-LAST_N:].mean(0)
    )  # MIND histories are in click order (latest last)
    kh = h[has_kg[h]]
    if len(kh):
        user_kg[p] = l2n(kg_raw[kh].mean(0))
    counts = np.bincount(art_cat[h], minlength=N_CAT).astype(np.float32)
    share = counts / counts.sum()
    user_aff[p] = share
    nz = share[share > 0]
    entropy[p] = float(-(nz * np.log(nz)).sum())
    diversity[p] = len(nz) / len(h)
    top_cat[p] = int(np.argmax(counts))

print(
    f"{P:,} profiles built in {time.time()-t0:.1f}s | cold-profile share of impressions: "
    f"{(beh.profile_idx == 0).mean():.2%}"
)


91,359 profiles built in 1.8s | cold-profile share of impressions: 2.37%


## 6. Candidate tables & the negative-sampling policy
**Training**: Each clicked article becomes one *group* = [positive, up to K=4 negatives]. Negatives are drawn from the same impression's non-clicked articles **without replacement** (seed 42). If an impression has fewer than 4 negatives, the group keeps the ones it has; the model masks the empty slots, so no negative is ever duplicated. Impressions without any negative are dropped from training.

**Val/Test**: One group per impression with the **full slate** of candidates (original order), so evaluation ranks exactly what the user saw. Impressions without a click are dropped (no metric is defined).


In [11]:
def build_groups(df, split):
    g_imp, g_slots = [], []
    if split == "train":
        for imp_id, c, l in zip(
            df.impression_id.values, df.cand_idx.values, df.cand_lab.values
        ):
            pos, neg = c[l == 1], c[l == 0]
            if len(neg) == 0:
                continue
            for pnews in pos:
                k = min(N_NEG, len(neg))
                g_imp.append(imp_id)
                g_slots.append(
                    (
                        np.concatenate(
                            [[pnews], rng.choice(neg, size=k, replace=False)]
                        ),
                        np.concatenate([[1], np.zeros(k, np.int8)]),
                    )
                )
    else:
        for imp_id, c, l in zip(
            df.impression_id.values, df.cand_idx.values, df.cand_lab.values
        ):
            if l.sum() == 0:
                continue
            g_imp.append(imp_id)
            g_slots.append((c, l))
    n = np.array([len(s[0]) for s in g_slots], dtype=np.int32)
    groups = pd.DataFrame(
        {
            "group_id": np.arange(len(g_imp)),
            "impression_id": g_imp,
            "cand_start": np.concatenate([[0], np.cumsum(n)[:-1]]),
            "n_cand": n,
        }
    )
    cands = pd.DataFrame(
        {
            "group_id": np.repeat(groups.group_id.values, n),
            "impression_id": np.repeat(groups.impression_id.values, n),
            "slot": np.concatenate([np.arange(k) for k in n]).astype(np.int16),
            "news_idx": np.concatenate([s[0] for s in g_slots]).astype(np.int32),
            "label": np.concatenate([s[1] for s in g_slots]).astype(np.int8),
        }
    )
    return groups, cands


tables = {}

for split in SPLITS:
    df = beh[beh.split == split]
    groups, cands = build_groups(df, split)
    tables[split] = (groups, cands)
    print(
        f"{split:5s}: {len(groups):>8,} groups | {len(cands):>10,} candidate rows | "
        f"mean slate {groups.n_cand.mean():.1f} | positives {int(cands.label.sum()):,}"
    )

g_tr = tables["train"][0]
print(
    f"Train groups with < {N_NEG} negatives (masked, not padded with repeats): "
    f"{(g_tr.n_cand < N_NEG + 1).sum():,} ({(g_tr.n_cand < N_NEG + 1).mean():.2%})"
)


train:  189,519 groups |    906,625 candidate rows | mean slate 4.8 | positives 189,519
val  :   30,270 groups |  1,222,429 candidate rows | mean slate 40.4 | positives 46,825


test :   73,152 groups |  2,740,998 candidate rows | mean slate 37.5 | positives 111,383
Train groups with < 4 negatives (masked, not padded with repeats): 19,075 (10.06%)


## 7. Candidate-level features (popularity + user × article)
* `cos_user_hist` = cos(`user_history_emb`, `article_content_emb`)
* `max_cos_hist` = highest cosine between the candidate and **any single** clicked article
* `cos_last5` = cosine with the mean of the 5 most recent clicks (short-term interest)
* `cat_affinity` = share of the user's clicks in the candidate's category
* `category_match` = 1 if the candidate's category is the user's top category
* `kg_match` = cos(user TransE profile, article TransE vector); NaN if either side has no entity

Undefined user–article similarities are filled from training statistics. Category match is zero for an empty history; `has_history` identifies that state.


In [12]:
imp_info = beh.set_index("impression_id")[["day", "profile_idx"]]


def rowwise_dot(A, ia, B, ib, chunk=100_000):
    out = np.empty(len(ia), dtype=np.float32)
    for s in range(0, len(ia), chunk):
        out[s : s + chunk] = np.einsum(
            "ij,ij->i", A[ia[s : s + chunk]], B[ib[s : s + chunk]]
        )
    return out


def add_candidate_features(cands):
    info = imp_info.loc[cands.impression_id.values]
    day = info["day"].values
    prof = info["profile_idx"].values
    nidx = cands["news_idx"].values
    cold_user = hist_len[prof] == 0
    # popularity (lagged)
    sd = stats_day(day)
    clk, ex = cum_clk[sd, nidx], cum_exp[sd, nidx]
    cands["hist_ctr"] = ((clk + CTR_ALPHA * CTR_PRIOR) / (ex + CTR_ALPHA)).astype(
        np.float32
    )
    cands["log_exposure"] = np.log1p(ex).astype(np.float32)
    cands["is_cold_article"] = (clk == 0).astype(np.int8)
    # user x article
    cands["cos_user_hist"] = rowwise_dot(user_emb, prof, content_emb, nidx)
    cands["cos_last5"] = rowwise_dot(user_last, prof, content_emb, nidx)
    cat = art_cat[nidx]
    cands["cat_affinity"] = user_aff[prof, cat]
    cm = (cat == top_cat[prof]).astype(np.float32)
    cm[cold_user] = 0.0
    cands["category_match"] = cm
    kgm = rowwise_dot(np.nan_to_num(user_kg), prof, np.nan_to_num(kg_raw), nidx)
    kgm[~(has_kg[nidx] & ~np.isnan(user_kg[prof, 0]))] = np.nan
    cands["kg_match"] = kgm
    # max cosine over the full history: one matrix product per (profile, group)
    mx = np.full(len(cands), np.nan, dtype=np.float32)
    order = np.argsort(prof, kind="stable")
    bounds = np.flatnonzero(np.diff(prof[order])) + 1
    for rows in np.split(order, bounds):
        h = hist_lists[prof[rows[0]]]
        if len(h):
            mx[rows] = (content_emb[nidx[rows]] @ content_emb[h].T).max(1)
    cands["max_cos_hist"] = mx
    return cands


for split in SPLITS:
    t0 = time.time()
    groups, cands = tables[split]
    tables[split] = (groups, add_candidate_features(cands))
    print(f"{split}: candidate features in {time.time()-t0:.1f}s")


train: candidate features in 3.8s


val: candidate features in 1.2s


test: candidate features in 5.1s


## 8. Impression-level context
* `hour_sin`, `hour_cos` of the impression hour
* `habit_score` (optional): cos(2π·Δt/24), Δt = hours since the user's previous impression (any earlier impression, always in the past). NaN for a user's first visit.


In [13]:
beh = beh.sort_values(["user_id", "time", "impression_id"], kind="mergesort")
gap_h = (
    beh["time"] - beh.groupby("user_id")["time"].shift(1)
).dt.total_seconds() / 3600.0
beh["habit_score"] = np.cos(2 * np.pi * gap_h / 24.0).astype(np.float32)
beh = beh.sort_values("impression_id").reset_index(drop=True)
hour = beh["time"].dt.hour + beh["time"].dt.minute / 60.0
beh["hour_sin"] = np.sin(2 * np.pi * hour / 24).astype(np.float32)
beh["hour_cos"] = np.cos(2 * np.pi * hour / 24).astype(np.float32)


## 9. Missing-value conventions

Absent mapped entities and empty-history content/category profiles are represented by zero vectors. Empty-history scalar profile values and category match are zero before standardization. Undefined interaction similarities and the optional habit score use training-fitted scalar means. Presence flags distinguish unavailable evidence from observed values.

In [14]:
CAND_NUM = [
    "hist_ctr",
    "log_exposure",
    "is_cold_article",
    "category_match",
    "cos_user_hist",
    "max_cos_hist",
    "cos_last5",
    "cat_affinity",
    "kg_match",
]
train_c = tables["train"][1]
impute_means = {
    c: float(np.nanmean(train_c[c].values.astype(np.float64))) for c in CAND_NUM
}
impute_means["category_match"] = 0.0
missing_report = {}

for split in SPLITS:
    g, c = tables[split]
    missing_report[split] = {k: int(c[k].isna().sum()) for k in CAND_NUM}
    for k in CAND_NUM:
        c[k] = c[k].fillna(impute_means[k]).astype(np.float32)

print("NaNs filled per split:")
print(pd.DataFrame(missing_report))
tr_imp = beh[beh.split == "train"]
tr_prof = tr_imp["profile_idx"].values
tr_prof_warm = tr_prof[hist_len[tr_prof] > 0]
entropy_mean = 0.0
diversity_mean = 0.0
cold_rows = hist_len == 0
user_emb[cold_rows] = 0.0
user_aff[cold_rows] = 0.0
entropy[cold_rows] = 0.0
diversity[cold_rows] = 0.0
kg_emb = kg_raw.copy()
kg_emb[~has_kg] = 0.0
kg_emb[0] = 0.0  # PAD row
habit_mean = float(np.nanmean(beh.loc[beh.split == "train", "habit_score"]))
beh["habit_score"] = beh["habit_score"].fillna(habit_mean).astype(np.float32)
impute_means.update(
    {
        "user_history_emb": "zero vector for empty history",
        "user_category_affinity": "zero vector for empty history",
        "reading_entropy": entropy_mean,
        "diversity_ratio": diversity_mean,
        "kg_entity_emb": "zero vector when no mapped entities",
        "habit_score": habit_mean,
    }
)

for k, v in impute_means.items():
    print(f"  {k:24s} {v if isinstance(v, str) else round(v, 5)}")

assert (
    not np.isnan(user_emb).any()
    and not np.isnan(kg_emb).any()
    and not np.isnan(user_aff).any()
)
hybrid_news = news.copy()
hybrid_news["cbf_row_idx"] = hybrid_news["news_idx"] - 1
hybrid_news.to_pickle(FEATURE_DIR / "hybrid_news.pkl")


NaNs filled per split:
                  train     val    test
hist_ctr              0       0       0
log_exposure          0       0       0
is_cold_article       0       0       0
category_match        0       0       0
cos_user_hist     17274   30575   82907
max_cos_hist      17274   30575   82907
cos_last5         17274   30575   82907
cat_affinity      17274   30575   82907
kg_match         188540  289390  646793
  hist_ctr                 0.04417
  log_exposure             2.58631
  is_cold_article          0.65208
  category_match           0.0
  cos_user_hist            0.18081
  max_cos_hist             0.26353
  cos_last5                0.12936
  cat_affinity             0.1744
  kg_match                 0.45148
  user_history_emb         zero vector for empty history
  user_category_affinity   zero vector for empty history
  reading_entropy          0.0
  diversity_ratio          0.0
  kg_entity_emb            zero vector when no mapped entities
  habit_score              0

## 10. Export (identical format for every split)
```
OUT_DIR/
  articles.parquet            news_idx, news_id, category_idx, subcategory_idx, has_kg_entity, in_train
  article_content_emb.npy     [N_ART, 384]   row 0 = PAD
  kg_entity_emb.npy           [N_ART, 100]   zero when no mapped entities
  user_profiles.parquet       profile_idx, history_length, log_history_length, has_history, reading_entropy, diversity_ratio
  user_history_emb.npy        [P, 384]
  user_category_affinity.npy  [P, N_CAT]
  {split}/impressions.parquet impression_id, orig_impression_id, source, user_id, time, day, profile_idx,
                              history_length, hour_sin, hour_cos, habit_score, n_pos, n_cand
  {split}/groups.parquet      group_id, impression_id, cand_start, n_cand
  {split}/candidates.parquet  group_id, impression_id, slot, news_idx, label, <9 candidate features>
  feature_meta.json           feature lists, vocab sizes, imputation means, split & cold-start definitions
```


In [15]:
articles = news[
    [
        "news_idx",
        "news_id",
        "category",
        "subcategory",
        "category_idx",
        "subcategory_idx",
        "has_kg_entity",
        "in_train",
    ]
].copy()
save_table(articles, "articles")
np.save(os.path.join(OUT_DIR, "kg_entity_emb.npy"), kg_emb.astype(np.float32))
profiles = pd.DataFrame(
    {
        "profile_idx": np.arange(P),
        "history_length": hist_len,
        "log_history_length": np.log1p(hist_len).astype(np.float32),
        "has_history": (hist_len > 0).astype(np.int8),
        "reading_entropy": entropy,
        "diversity_ratio": diversity,
    }
)
save_table(profiles, "user_profiles")
np.save(os.path.join(OUT_DIR, "user_history_emb.npy"), user_emb.astype(np.float32))
np.save(
    os.path.join(OUT_DIR, "user_category_affinity.npy"), user_aff.astype(np.float32)
)
IMP_COLS = [
    "impression_id",
    "orig_impression_id",
    "source",
    "user_id",
    "time",
    "day",
    "profile_idx",
    "hour_sin",
    "hour_cos",
    "habit_score",
    "n_pos",
    "n_cand",
]

for split in SPLITS:
    g, c = tables[split]
    imp = beh.loc[beh.impression_id.isin(g.impression_id.unique()), IMP_COLS].copy()
    imp["history_length"] = hist_len[imp.profile_idx.values]
    save_table(imp, f"{split}/impressions")
    save_table(g, f"{split}/groups")
    save_table(c, f"{split}/candidates")


In [16]:
meta = {
    "split": {
        "scheme": "5-1-1 by calendar day",
        "first_day": str(first_day.date()),
        "train_days": TRAIN_DAYS,
        "val_day": VAL_DAY,
        "test_day": TEST_DAY,
        "summary": {
            s: {
                "impressions": int(split_summary.loc[s, "impressions"]),
                "start": str(split_summary.loc[s, "start"]),
                "end": str(split_summary.loc[s, "end"]),
            }
            for s in SPLITS
        },
    },
    "negative_sampling": {
        "train": f"per positive, up to {N_NEG} negatives from same impression, no replacement, "
        f"seed {SEED}; short groups masked (no repeats); impressions without negatives dropped",
        "val_test": "full impression slate; impressions without clicks dropped",
    },
    "cold_start": {
        "sparse_user_max": SPARSE_USER_MAX,
        "cold_user": "history_length == 0",
        "sparse_user": f"1 <= history_length < {SPARSE_USER_MAX}",
        "cold_article": "no clicks in any impression before the impression day (is_cold_article == 1)",
    },
    "leakage": {
        "popularity": "lagged by day (days < impression day)",
        "ctr_prior": CTR_PRIOR,
        "ctr_alpha": CTR_ALPHA,
        "stats_use_val_for_test": STATS_USE_VAL_FOR_TEST,
        "vocab_and_means": "training period only",
    },
    "excluded_features": ["slate_length", "article_recency", "session_dow"],
    "features": {
        "article": {
            "dense": ["article_content_emb", "kg_entity_emb"],
            "flags": ["has_kg_entity"],
            "categorical": ["category_idx", "subcategory_idx"],
        },
        "popularity": ["hist_ctr", "log_exposure", "is_cold_article"],
        "user": {
            "dense": ["user_history_emb"],
            "scalar": ["log_history_length", "has_history", "reading_entropy"],
            "category_profile": "user_category_affinity",
        },
        "user_x_article": [
            "category_match",
            "cos_user_hist",
            "max_cos_hist",
            "cos_last5",
            "cat_affinity",
            "kg_match",
        ],
        "context": ["hour_sin", "hour_cos"],
        "optional": {"user": ["diversity_ratio"], "context": ["habit_score"]},
    },
    "dims": {
        "text": int(D_TEXT),
        "kg": int(D_KG),
        "n_cat": int(N_CAT),
        "n_subcat": int(N_SUBCAT),
        "n_articles": int(N_ART),
        "n_profiles": int(P),
    },
    "imputation_means": impute_means,
    "config": {
        "user_pooling": USER_POOLING,
        "last_n": LAST_N,
        "text_model": TEXT_MODEL,
    },
}


In [17]:
meta["source_files"] = source_fingerprint(PATHS)

with open(os.path.join(OUT_DIR, "feature_meta.json"), "w") as f:
    json.dump(meta, f, indent=2, default=str)

print("Saved to", OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))


Saved to /mnt/data/KHTN2023/Project/bt4222-group12/data/processed/features
['article_content_emb.npy', 'article_content_emb_config.json', 'article_content_emb_ids.json', 'articles.parquet', 'feature_meta.json', 'hybrid_news.pkl', 'hybrid_user_state.pkl', 'kg_entity_emb.npy', 'mlp_inputs.pkl', 'nrms_features.pkl', 'test', 'train', 'user_category_affinity.npy', 'user_history_emb.npy', 'user_history_indices.npy', 'user_profiles.parquet', 'val']


## 11. Leakage & sanity checks


In [18]:
day_flat = np.repeat(beh["day"].values, beh["n_cand"].values)
cand_flat = np.concatenate(beh["cand_idx"].tolist())
lab_flat = np.concatenate(beh["cand_lab"].tolist())
c = tables["test"][1].sample(min(25, len(tables["test"][1])), random_state=SEED)

for _, r in c.iterrows():
    d = int(imp_info.loc[r.impression_id, "day"])
    m = (day_flat < stats_day(d)) & (cand_flat == r.news_idx)
    assert (
        abs(np.log1p(m.sum()) - r.log_exposure) < 1e-4
    ), "exposure count leak/mismatch"
    assert (
        int(lab_flat[m].sum() == 0) == r.is_cold_article
    ), "cold-article flag mismatch"

del day_flat, cand_flat, lab_flat
print(
    "(a) lagged popularity matches a raw recount (days strictly before the impression day)"
)

for d in range(7):
    assert stats_day(d) <= d  # cum_*[stats_day(d)] covers days < stats_day(d) <= d

print("(b) every split only uses popularity from days strictly before its own day")
assert set(cat_vocab) - {"<PAD>", "<UNK>"} <= set(news.loc[news.in_train, "category"])
print("(c) category vocabulary built from training-period articles only")
summary = []

for split in SPLITS:
    g, c = tables[split]
    imp = beh[beh.impression_id.isin(g.impression_id)]
    hl = hist_len[imp.profile_idx.values]
    summary.append(
        {
            "split": split,
            "groups": len(g),
            "rows": len(c),
            "cold_user_imps": f"{(hl == 0).mean():.2%}",
            "sparse_user_imps": f"{((hl > 0) & (hl < SPARSE_USER_MAX)).mean():.2%}",
            "cold_article_rows": f"{c.is_cold_article.mean():.2%}",
            "cold_article_positives": f'{c.loc[c.label == 1, "is_cold_article"].mean():.2%}',
        }
    )

print(pd.DataFrame(summary).to_string(index=False))
print("\nCandidate feature summary (train):")
print(tables["train"][1][CAND_NUM].describe().T[["mean", "std", "min", "max"]].round(4))


(a) lagged popularity matches a raw recount (days strictly before the impression day)
(b) every split only uses popularity from days strictly before its own day
(c) category vocabulary built from training-period articles only
split  groups    rows cold_user_imps sparse_user_imps cold_article_rows cold_article_positives
train  189519  906625          1.95%           11.33%            65.21%                 66.35%
  val   30270 1222429          2.55%           10.73%            60.33%                 65.71%
 test   73152 2740998          3.03%           11.06%            54.86%                 51.36%

Candidate feature summary (train):
                   mean     std     min     max
hist_ctr         0.0442  0.0310  0.0010  0.4545
log_exposure     2.5863  2.8908  0.0000  9.8130
is_cold_article  0.6521  0.4763  0.0000  1.0000
category_match   0.2412  0.4278  0.0000  1.0000
cos_user_hist    0.1808  0.1177 -0.2431  1.0000
max_cos_hist     0.2635  0.1278 -0.1982  1.0000
cos_last5        0.129

## 12. Standardized feature arrays and history sequences

Fit the existing normalization on training groups and export identical input arrays for the ranking variants. Attention histories retain the most recent 50 article indices per profile.


In [19]:
data = {
    s: {
        "imp": load_table(FEATURE_DIR / f"{s}/impressions"),
        "groups": load_table(FEATURE_DIR / f"{s}/groups"),
        "cands": load_table(FEATURE_DIR / f"{s}/candidates"),
    }
    for s in SPLITS
}
SPARSE_MAX = meta["cold_start"]["sparse_user_max"]
F = meta["features"]
POP_COLS = F["popularity"]  # hist_ctr, log_exposure, is_cold_article
INTER_COLS = F["user_x_article"]  # 6 user x article features
CAND_COLS = POP_COLS + INTER_COLS
OPTIONAL_USER = F["optional"]["user"]  # diversity_ratio
OPTIONAL_CTX = F["optional"]["context"]  # habit_score
USER_COLS = F["user"]["scalar"] + OPTIONAL_USER  # all user scalars stored in user_x
CTX_COLS = F["context"] + OPTIONAL_CTX  # all context features stored in ctx_x
OPTIONAL_FEATURES = OPTIONAL_USER + OPTIONAL_CTX
BINARY = {"is_cold_article", "category_match", "has_history"}
used = set(CAND_COLS + USER_COLS + CTX_COLS) | {
    "article_content_emb",
    "kg_entity_emb",
    "category_idx",
    "subcategory_idx",
}
assert not used & set(meta["excluded_features"]), "contract: excluded feature in use"
tr_c = data["train"]["cands"]
tr_prof = (
    data["train"]["imp"]
    .set_index("impression_id")
    .loc[data["train"]["groups"].impression_id, "profile_idx"]
    .values
)
cand_mu = {c: (0.0 if c in BINARY else float(tr_c[c].mean())) for c in CAND_COLS}
cand_sd = {c: (1.0 if c in BINARY else float(tr_c[c].std() or 1.0)) for c in CAND_COLS}
user_mu = {
    c: (0.0 if c in BINARY else float(profiles[c].values[tr_prof].mean()))
    for c in USER_COLS
}
user_sd = {
    c: (1.0 if c in BINARY else float(profiles[c].values[tr_prof].std() or 1.0))
    for c in USER_COLS
}
ctx_mu = {
    c: float(data["train"]["imp"][c].mean()) if c == "habit_score" else 0.0
    for c in CTX_COLS
}
ctx_sd = {
    c: float(data["train"]["imp"][c].std() or 1.0) if c == "habit_score" else 1.0
    for c in CTX_COLS
}
user_x_all = np.stack(
    [(profiles[c].values - user_mu[c]) / user_sd[c] for c in USER_COLS], 1
).astype(np.float32)


In [20]:
def assemble(split):
    d = data[split]
    g = d["groups"].sort_values("group_id").reset_index(drop=True)
    c = d["cands"].sort_values(["group_id", "slot"]).reset_index(drop=True)
    imp = d["imp"].set_index("impression_id").loc[g.impression_id]
    start = np.concatenate([[0], np.cumsum(g.n_cand.values)[:-1]]).astype(np.int64)
    assert (
        len(c) == g.n_cand.sum()
        and (c.group_id.values == np.repeat(g.group_id.values, g.n_cand.values)).all()
    )
    prof = imp["profile_idx"].values.astype(np.int64)
    return {
        "split": split,
        "start": start,
        "n": g.n_cand.values.astype(np.int64),
        "impression_id": g.impression_id.values,
        "user_id": imp["user_id"].values,
        "orig_impression_id": imp["orig_impression_id"].values,
        "source": imp["source"].values,
        "hist_len": imp["history_length"].values,
        "news": c.news_idx.values.astype(np.int64),
        "label": c.label.values.astype(np.float32),
        "slot": c.slot.values,
        "is_cold_article": c.is_cold_article.values.astype(np.int8),
        "cand_x": np.stack(
            [(c[k].values - cand_mu[k]) / cand_sd[k] for k in CAND_COLS], 1
        ).astype(np.float32),
        "prof": prof,
        "user_x": user_x_all[prof],
        "ctx_x": np.stack(
            [(imp[k].values - ctx_mu[k]) / ctx_sd[k] for k in CTX_COLS], 1
        ).astype(np.float32),
    }


A = {s: assemble(s) for s in SPLITS}
assert A["train"]["n"].max() <= 5

for s in SPLITS:
    hl = A[s]["hist_len"]
    print(
        f"{s:5s}: cold users {np.mean(hl == 0):.2%} | sparse (1-{SPARSE_MAX-1}) {np.mean((hl > 0) & (hl < SPARSE_MAX)):.2%} "
        f"| cold-article candidates {A[s]['is_cold_article'].mean():.2%}"
    )

prepared_inputs = {
    "A": A,
    "cand_mu": cand_mu,
    "cand_sd": cand_sd,
    "user_mu": user_mu,
    "user_sd": user_sd,
    "ctx_mu": ctx_mu,
    "ctx_sd": ctx_sd,
    "user_x_all": user_x_all,
}
pd.to_pickle(prepared_inputs, FEATURE_DIR / "mlp_inputs.pkl")
HIST_LEN = 50
HIST = np.zeros((len(hist_lists), HIST_LEN), np.int64)

for profile_idx, history in enumerate(hist_lists):
    tail = history[-HIST_LEN:]
    HIST[profile_idx, : len(tail)] = tail

assert (HIST[0] == 0).all()
np.save(FEATURE_DIR / "user_history_indices.npy", HIST)


train: cold users 1.91% | sparse (1-4) 9.57% | cold-article candidates 65.21%
val  : cold users 2.55% | sparse (1-4) 10.73% | cold-article candidates 60.33%
test : cold users 3.03% | sparse (1-4) 11.06% | cold-article candidates 54.86%


## 13. Title sequences and historical popularity sets

Prepare padded title tokens, aligned word vectors and previous-day category-stratified article sets. These sequences are consumed by trainable attention encoders.


In [21]:
def daily_click_counts(df, news_id2idx):
    counts = defaultdict(Counter)
    for row in df.itertuples(index=False):
        day = row.time.normalize()
        for item in row.impressions.split():
            if item.endswith("-1"):
                idx = news_id2idx.get(item[:-2])
                if idx:
                    counts[day][idx] += 1
    return counts


def build_popular_set(click_counts, news_cat, per_cat=2, extra=10, min_cat_clicks=20):
    """
    Every category with >= min_cat_clicks clicks contributes its top `per_cat` articles;
    `extra` additional slots are allocated proportionally to category click volume (largest remainder).
    """
    cat_clicks = Counter()
    per_cat_items = defaultdict(list)
    for idx, c in click_counts.items():
        cat = news_cat[idx]
        cat_clicks[cat] += c
        per_cat_items[cat].append((c, idx))
    eligible = sorted(
        [c for c, n in cat_clicks.items() if n >= min_cat_clicks],
        key=lambda c: -cat_clicks[c],
    )
    if not eligible:
        return [], {}
    total = sum(cat_clicks[c] for c in eligible)
    quotas = {c: extra * cat_clicks[c] / total for c in eligible}
    alloc = {c: per_cat + int(quotas[c]) for c in eligible}
    leftover = extra - sum(int(q) for q in quotas.values())
    for c in sorted(eligible, key=lambda c: quotas[c] - int(quotas[c]), reverse=True)[
        :leftover
    ]:
        alloc[c] += 1
    ids = []
    for c in eligible:
        top = sorted(per_cat_items[c], reverse=True)[: alloc[c]]
        alloc[c] = len(top)
        ids += [idx for _, idx in top]
    return ids, alloc


In [22]:
def prepare_data(
    data_dir, device, pop_per_cat=2, pop_extra=10, pop_min_cat_clicks=20, debug=False
):
    """
    Loads MIND-small, builds the title lookup / GloVe matrix, the strict 5-1-1 temporal split,
    the daily popular sets, and the validation/test records. Shared by training and evaluation scripts.
    """
    # 4.1 Load and Merge News Catalogue
    print("\n[1/7] Loading and Preprocessing News Catalogue...")
    t0 = time.time()
    _, all_news = load_preprocessed(PATHS)
    all_news = all_news.copy()

    def tokenize(text):
        if not isinstance(text, str):
            return []
        return re.findall(r"\w+|[^\w\s]", text.lower())

    word_counts = Counter()
    for title in all_news["title"]:
        word_counts.update(tokenize(title))
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, count in word_counts.items():
        if count >= 2:
            vocab[word] = len(vocab)
    print(f"Vocabulary size (min_freq >= 2): {len(vocab)} words.")
    news_id2idx = {"<PAD>": 0}
    news_cat = ["<PAD>"]
    for nid, cat in zip(all_news["news_id"], all_news["category"]):
        news_id2idx[nid] = len(news_id2idx)
        news_cat.append(cat)
    max_title_len = 30
    lookup = np.zeros((len(news_id2idx), max_title_len), dtype=np.int64)
    for row in all_news.itertuples(index=False):
        idx = news_id2idx[row.news_id]
        tokens = [vocab.get(w, 1) for w in tokenize(row.title)][:max_title_len]
        lookup[idx, : len(tokens)] = tokens
    news_title_lookup = torch.tensor(lookup, dtype=torch.long, device=device)
    print(
        f"News lookup tensor: {news_title_lookup.shape} (Time: {time.time()-t0:.2f}s)"
    )
    # 4.2 Load Pretrained GloVe Embeddings
    print("\n[2/7] Loading GloVe 300d Pretrained Word Embeddings...")
    t0 = time.time()
    matrix = np.random.normal(scale=0.1, size=(len(vocab), 300)).astype(np.float32)
    matrix[0] = 0.0  # PAD
    matched = 0
    with open(PATHS["glove"], "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            w = parts[0]
            if w in vocab:
                matrix[vocab[w]] = np.asarray(parts[1:], dtype=np.float32)
                matched += 1
    pretrained_embeddings = torch.tensor(matrix, dtype=torch.float32)
    print(
        f"GloVe coverage: {matched}/{len(vocab)} ({matched/len(vocab)*100:.2f}%) in {time.time()-t0:.2f}s."
    )
    # 4.3 Load Behaviors and Strict Temporal Split
    print("\n[3/7] Parsing Behaviors and Creating Strict Temporal Partitions...")
    t0 = time.time()
    beh, _ = load_preprocessed(PATHS)
    train_b = beh[beh.source == "train"].copy()
    dev_b = beh[beh.source == "dev"].copy()
    train_df = beh[beh.split == "train"].copy()
    val_df = beh[beh.split == "val"].copy()
    test_df = beh[beh.split == "test"].copy()
    if debug:
        train_df = train_df.sample(n=min(4000, len(train_df)), random_state=0)
        val_df = val_df.sample(n=min(1500, len(val_df)), random_state=0)
        test_df = test_df.sample(n=min(3000, len(test_df)), random_state=0)
    print(f"Temporal Holdout Splits:")
    print(
        f"  • Train (Days 1–5):       {len(train_df):>7} impressions ({train_df['time'].min()} to {train_df['time'].max()})"
    )
    print(
        f"  • Validation (Day 6):     {len(val_df):>7} impressions ({val_df['time'].min()} to {val_df['time'].max()})"
    )
    print(
        f"  • Test (Day 7):           {len(test_df):>7} impressions ({test_df['time'].min()} to {test_df['time'].max()})"
    )
    # 4.4 Popularity branch: for an impression on day d, use clicks logged on day d-1 (no same-day leakage).
    # Day 1 has no previous logged day, so it falls back to its own clicks. Clicks come from train_b only (Days 1-6).
    all_days = sorted(
        set(train_b["time"].dt.normalize()) | set(dev_b["time"].dt.normalize())
    )
    day2idx = {d: i for i, d in enumerate(all_days)}
    clicks_by_day = daily_click_counts(train_b, news_id2idx)
    pop_lists = []
    print("\nDaily category-stratified popular sets (source day -> size, categories):")
    for d in all_days:
        src = d - pd.Timedelta(days=1)
        if src not in clicks_by_day:
            src = d
        ids, alloc = build_popular_set(
            clicks_by_day[src], news_cat, pop_per_cat, pop_extra, pop_min_cat_clicks
        )
        pop_lists.append(ids)
        print(
            f"  • {d.date()} <- {src.date()}: {len(ids)} articles over {len(alloc)} categories {dict(alloc)}"
        )
    K = max(len(p) for p in pop_lists)
    pop_sets = torch.zeros(len(all_days), K, dtype=torch.long)
    for i, ids in enumerate(pop_lists):
        pop_sets[i, : len(ids)] = torch.tensor(ids, dtype=torch.long)
    pop_sets = pop_sets.to(device)
    max_hist_len = 50

    def parse_history(raw):
        raw_h = raw.split() if isinstance(raw, str) else []
        hist = [news_id2idx[x] for x in raw_h if x in news_id2idx][-max_hist_len:]
        return [0] * (max_hist_len - len(hist)) + hist, len(raw_h)

    def build_eval_records(df):
        records = []
        for row in df.itertuples(index=False):
            hist, hist_len = parse_history(row.history)
            cands = []
            labels = []
            for item in row.impressions.split():
                nid, lbl = item.split("-")
                if nid in news_id2idx:
                    cands.append(news_id2idx[nid])
                    labels.append(int(lbl))
            if sum(labels) > 0 and len(labels) > sum(labels):
                records.append(
                    {
                        "impression_id": row.impression_id,
                        "timestamp": row.time,
                        "day_idx": day2idx[row.time.normalize()],
                        "history": hist,
                        "history_len": hist_len,
                        "candidates": cands,
                        "labels": labels,
                    }
                )
        return records

    val_records = build_eval_records(val_df)
    test_records = build_eval_records(test_df)
    print(
        f"\nConstructed {len(val_records)} validation records and {len(test_records)} test records in {time.time()-t0:.2f}s."
    )
    return SimpleNamespace(
        all_news=all_news,
        tokenize=tokenize,
        max_title_len=max_title_len,
        news_id2idx=news_id2idx,
        news_cat=news_cat,
        news_title_lookup=news_title_lookup,
        pretrained_embeddings=pretrained_embeddings,
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        all_days=all_days,
        day2idx=day2idx,
        pop_lists=pop_lists,
        pop_sets=pop_sets,
        max_hist_len=max_hist_len,
        parse_history=parse_history,
        val_records=val_records,
        test_records=test_records,
    )


In [23]:
random.seed(SEED)
np.random.seed(SEED)
NRMS_POP_PER_CAT = 2
NRMS_POP_EXTRA = 10
NRMS_POP_MIN_CAT_CLICKS = 20
nrms_data = prepare_data(
    str(PATHS["raw"]),
    torch.device("cpu"),
    NRMS_POP_PER_CAT,
    NRMS_POP_EXTRA,
    NRMS_POP_MIN_CAT_CLICKS,
)
nrms_payload = {
    key: (value.numpy() if isinstance(value, torch.Tensor) else value)
    for key, value in vars(nrms_data).items()
    if not callable(value)
}
nrms_payload["news_id2idx"] = news_id2idx
nrms_payload["popularity_config"] = {
    "per_cat": NRMS_POP_PER_CAT,
    "extra": NRMS_POP_EXTRA,
    "min_cat_clicks": NRMS_POP_MIN_CAT_CLICKS,
}
global_click_counts = {}

for row in nrms_data.train_df.sort_values("orig_impression_id").itertuples(index=False):
    for item in row.impressions.split():
        news_id, label = item.rsplit("-", 1)
        if label == "1" and news_id in news_id2idx:
            article_idx = news_id2idx[news_id]
            global_click_counts[article_idx] = (
                global_click_counts.get(article_idx, 0) + 1
            )

nrms_payload["top_popular_ids"] = [
    idx
    for idx, _ in sorted(
        global_click_counts.items(), key=lambda entry: entry[1], reverse=True
    )[:20]
]
pd.to_pickle(nrms_payload, FEATURE_DIR / "nrms_features.pkl")



[1/7] Loading and Preprocessing News Catalogue...


Vocabulary size (min_freq >= 2): 21351 words.


News lookup tensor: torch.Size([65239, 30]) (Time: 0.94s)

[2/7] Loading GloVe 300d Pretrained Word Embeddings...


GloVe coverage: 20674/21351 (96.83%) in 3.12s.

[3/7] Parsing Behaviors and Creating Strict Temporal Partitions...


Temporal Holdout Splits:
  • Train (Days 1–5):        126695 impressions (2019-11-09 00:00:19 to 2019-11-13 23:59:21)
  • Validation (Day 6):       30270 impressions (2019-11-14 00:00:00 to 2019-11-14 23:59:13)
  • Test (Day 7):             73152 impressions (2019-11-15 00:00:01 to 2019-11-15 23:58:03)



Daily category-stratified popular sets (source day -> size, categories):
  • 2019-11-09 <- 2019-11-09: 38 articles over 14 categories {'news': 5, 'tv': 3, 'lifestyle': 3, 'sports': 3, 'movies': 3, 'finance': 3, 'foodanddrink': 3, 'health': 3, 'travel': 2, 'music': 2, 'autos': 2, 'entertainment': 2, 'weather': 2, 'video': 2}
  • 2019-11-10 <- 2019-11-09: 38 articles over 14 categories {'news': 5, 'tv': 3, 'lifestyle': 3, 'sports': 3, 'movies': 3, 'finance': 3, 'foodanddrink': 3, 'health': 3, 'travel': 2, 'music': 2, 'autos': 2, 'entertainment': 2, 'weather': 2, 'video': 2}
  • 2019-11-11 <- 2019-11-10: 38 articles over 14 categories {'news': 6, 'music': 4, 'sports': 3, 'lifestyle': 3, 'finance': 3, 'movies': 3, 'tv': 2, 'foodanddrink': 2, 'health': 2, 'autos': 2, 'entertainment': 2, 'travel': 2, 'video': 2, 'weather': 2}
  • 2019-11-12 <- 2019-11-11: 38 articles over 14 categories {'news': 5, 'sports': 4, 'tv': 3, 'lifestyle': 3, 'finance': 3, 'music': 3, 'entertainment': 3, 'travel': 


Constructed 30270 validation records and 73152 test records in 3.26s.


## 14. Impression context export


In [24]:
state_beh, _ = load_preprocessed(PATHS)
state_beh["profile_idx"] = uniq_hist.get_indexer(state_beh["history"].str.strip())
assert (state_beh["profile_idx"] >= 0).all()
ustate = state_beh[["impression_id", "user_id", "time"]].copy()
ustate["log_history_length"] = np.log1p(hist_len[state_beh["profile_idx"].values])
ustate["reading_entropy"] = entropy[state_beh["profile_idx"].values]
ustate = ustate.sort_values(["user_id", "time"])
ustate["inactivity_gap"] = (
    ustate.groupby("user_id")["time"].diff().dt.total_seconds() / 3600
)
ustate["first_imp"] = ustate["inactivity_gap"].isna().astype(np.int8)
hour = ustate["time"].dt.hour
ustate["hour_sin"] = np.sin(2 * np.pi * hour / 24)
ustate["hour_cos"] = np.cos(2 * np.pi * hour / 24)
ustate.drop(columns=["user_id", "time"]).to_pickle(
    FEATURE_DIR / "hybrid_user_state.pkl"
)


## 15. Completed feature bundle

Publish a content manifest after all exports succeed. Models verify these files independently of raw-data location. Use a new output directory when changing the prepared bundle.


In [25]:
# Publish the bundle only after every table and model input has been exported.
notebook_source = json.loads(
    (PROJECT_ROOT / "data_preparation/3_feature_engineering.ipynb").read_text(encoding="utf-8")
)
preparation = {
    "source_files": source_fingerprint(PATHS),
    "feature_code": content_identity([
        "".join(cell["source"]) for cell in notebook_source["cells"]
        if cell["cell_type"] == "code"
    ]),
    "data_utils_code": file_digest(PROJECT_ROOT / "utils/data_utils.py"),
    "embedding_identity": embedding_key,
    "glove": file_digest(PATHS["glove"]),
    "settings": meta,
}
feature_id = publish_features(PATHS, preparation)
print("Complete feature bundle:", feature_id)


Complete feature bundle: 0b88c245c7956d5501a46ac3bc54df1fab01de066e0213a478ccd88fe4ab856e
